# Relatório de Pré-processamento de Dados: Spotify Tracks Dataset

**Disciplina:** Tópicos Avançados em Computação IV  
**Projeto:** Pré-processamento de Dados em Ciência de Dados  

**Participantes:**  
- Adriano André da Silva
- Gustavo Henrique de Brum
- Guilherme Júnior Machado
- João Pedro Oliveira Barbosa
- Jorge Miguel Eberhard da Conceição

---

Este notebook apresenta a leitura, inspeção, pré-processamento e análise exploratória do dataset **Spotify Tracks Dataset**, disponibilizado publicamente no Kaggle. O objetivo é identificar problemas nos dados, aplicar técnicas de limpeza e transformação, calcular estatísticas descritivas e analisar relações entre atributos musicais, de forma a obter um conjunto de dados mais consistente e adequado para análise.

In [2]:
!pip install pandas numpy matplotlib seaborn scikit-learn

## 0 - Imports e Configurações Visuais

In [ ]:
import os
import matplotlib

matplotlib.use("Agg")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings

warnings.filterwarnings("ignore")

os.makedirs("resultados", exist_ok=True)

# Configurações visuais
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

## 1 - Leitura do Dataset

In [ ]:
# ============================================================
# 1. LEITURA DO DATASET
# ============================================================
print("=" * 60)
print("1. LEITURA E EXPLORAÇÃO DO DATASET")
print("=" * 60)

df = pd.read_csv("dataset.csv")

print(f"\nShape: {df.shape[0]} registros x {df.shape[1]} atributos")
print("\nPrimeiras linhas:")
print(df.head())

print("\nNomes das colunas:")
print(df.columns.tolist())

1. LEITURA E EXPLORAÇÃO DO DATASET

Shape: 114000 registros x 22 atributos

Primeiras linhas:
   Unnamed: 0.1  Unnamed: 0                track_id                 artists  \
0             0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1             1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2             2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3             3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4             4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Com

## 2 - Análise Inicial dos Dados

Nesta etapa, é realizada uma inspeção inicial do dataset para compreender sua estrutura e identificar possíveis problemas antes do pré-processamento. São verificados os tipos de variáveis, a presença de valores ausentes, a existência de registros duplicados e um resumo estatístico inicial das colunas numéricas. Essa análise ajuda a definir quais tratamentos serão necessários nas próximas etapas.

In [ ]:
# ============================================================
# 2. ANÁLISE INICIAL DOS DADOS
# ============================================================
print("\n" + "=" * 70)
print("2. ANÁLISE INICIAL DOS DADOS")
print("=" * 70)

# ------------------------------------------------------------
# Tipos de variáveis
# ------------------------------------------------------------
print("\n[1] Tipos de variáveis")
print("-" * 70)

tipos_df = pd.DataFrame({"Coluna": df.columns, "Tipo": df.dtypes.astype(str).values})
print(tipos_df.to_string(index=False))

# ------------------------------------------------------------
# Valores ausentes
# ------------------------------------------------------------
print("\n[2] Valores ausentes")
print("-" * 70)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame(
    {
        "Coluna": missing.index,
        "Ausentes": missing.values,
        "Percentual (%)": missing_pct.values,
    }
)

missing_df = missing_df[missing_df["Ausentes"] > 0].sort_values(
    by="Ausentes", ascending=False
)

if not missing_df.empty:
    print(
        missing_df.to_string(
            index=False, formatters={"Percentual (%)": "{:.4f}".format}
        )
    )
else:
    print("Nenhum valor ausente encontrado.")

# ------------------------------------------------------------
# Duplicatas
# ------------------------------------------------------------
print("\n[3] Duplicatas")
print("-" * 70)

duplicatas = df.duplicated(subset=["track_name", "artists"]).sum()
print(f"Registros duplicados considerando ['track_name', 'artists']: {duplicatas}")

# ------------------------------------------------------------
# Estatísticas descritivas iniciais
# ------------------------------------------------------------
print("\n[4] Estatísticas descritivas iniciais")
print("-" * 70)

desc = df.describe().T
desc = desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
print(desc.to_string(float_format=lambda x: f"{x:.4f}"))

# ------------------------------------------------------------
# Gráfico de valores ausentes
# ------------------------------------------------------------
print("\n[5] Gráfico de valores ausentes")
print("-" * 70)

fig, ax = plt.subplots(figsize=(10, 4))
missing_plot = missing[missing > 0]

if len(missing_plot) > 0:
    missing_plot.plot(kind="bar", ax=ax, color="salmon", edgecolor="black")
    ax.set_title("Valores Ausentes por Coluna")
    ax.set_ylabel("Quantidade")
    ax.set_xlabel("Coluna")
    plt.xticks(rotation=45, ha="right")
else:
    ax.text(
        0.5,
        0.5,
        "Nenhum valor ausente encontrado!",
        ha="center",
        va="center",
        fontsize=14,
        color="green",
        transform=ax.transAxes,
    )
    ax.set_title("Valores Ausentes por Coluna")
plt.tight_layout()
plt.savefig("resultados/01_valores_ausentes.png")
print("Gráfico salvo em /resultados: 01_valores_ausentes.png")


2. ANÁLISE INICIAL DOS DADOS

[1] Tipos de variáveis
----------------------------------------------------------------------
          Coluna    Tipo
    Unnamed: 0.1   int64
      Unnamed: 0   int64
        track_id     str
         artists     str
      album_name     str
      track_name     str
      popularity   int64
     duration_ms   int64
        explicit    bool
    danceability float64
          energy float64
             key   int64
        loudness float64
            mode   int64
     speechiness float64
    acousticness float64
instrumentalness float64
        liveness float64
         valence float64
           tempo float64
  time_signature   int64
     track_genre     str

[2] Valores ausentes
----------------------------------------------------------------------
    Coluna  Ausentes Percentual (%)
   artists         1         0.0009
album_name         1         0.0009
track_name         1         0.0009

[3] Duplicatas
-----------------------------------------------

## 3 - Pré-processamento dos Dados

Nesta etapa, o dataset passa por um conjunto de transformações para reduzir ruídos, corrigir problemas de qualidade e preparar a base para as análises estatísticas posteriores. São realizadas operações de remoção de duplicatas, tratamento de valores ausentes, validação de intervalos, transformação de atributos, tratamento de outliers e normalização de variáveis numéricas.

O objetivo desse processo é obter um conjunto de dados mais consistente, comparável e adequado para a interpretação dos resultados.

In [ ]:
# ============================================================
# 3. PRÉ-PROCESSAMENTO DOS DADOS
# ============================================================

print("\n" + "═" * 72)
print("3. PRÉ-PROCESSAMENTO DOS DADOS")
print("═" * 72)

df_clean = df.copy()
total_registros_inicial = len(df_clean)
print(
    f"\nDimensão inicial do dataset: {df_clean.shape[0]} linhas x {df_clean.shape[1]} colunas"
)

# ------------------------------------------------------------
# 3.1 Remoção de duplicatas
# ------------------------------------------------------------
print("\n[3.1] Remoção de duplicatas")
print("-" * 72)

antes = len(df_clean)
df_clean.drop_duplicates(subset=["track_name", "artists"], inplace=True)
removidos_duplicatas = antes - len(df_clean)

print(f"Critério utilizado: combinação de ['track_name', 'artists']")
print(f"Registros removidos: {removidos_duplicatas}")
print(f"Registros restantes: {len(df_clean)}")

# ------------------------------------------------------------
# 3.2 Tratamento de valores ausentes
# ------------------------------------------------------------
print("\n[3.2] Tratamento de valores ausentes")
print("-" * 72)

num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

ausentes_por_linha = df_clean.isnull().sum(axis=1)
removidos_ausentes = (ausentes_por_linha >= 3).sum()
df_clean = df_clean[ausentes_por_linha < 3].copy()

print(f"Registros removidos por 3 ou mais valores ausentes: {removidos_ausentes}")

imputacoes_realizadas = []

for col in num_cols:
    if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
        mediana = df_clean[col].median()
        qtd = df_clean[col].isnull().sum()
        df_clean[col].fillna(mediana, inplace=True)
        imputacoes_realizadas.append(
            f"Numérica  | {col:<18} | {qtd:>4} preenchidos com mediana ({mediana:.4f})"
        )

for col in cat_cols:
    if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
        moda = df_clean[col].mode()[0]
        qtd = df_clean[col].isnull().sum()
        df_clean[col].fillna(moda, inplace=True)
        imputacoes_realizadas.append(
            f"Categórica| {col:<18} | {qtd:>4} preenchidos com moda ('{moda}')"
        )

if imputacoes_realizadas:
    print("\nImputações realizadas:")
    for linha in imputacoes_realizadas:
        print(" - " + linha)
else:
    print("Nenhuma imputação foi necessária após a remoção de registros críticos.")

print(f"Valores ausentes restantes no dataset: {df_clean.isnull().sum().sum()}")

# ------------------------------------------------------------
# 3.3 Conversão de duração
# ------------------------------------------------------------
print("\n[3.3] Transformação de atributos")
print("-" * 72)

if "duration_ms" in df_clean.columns:
    df_clean["duration_min"] = df_clean["duration_ms"] / 60000
    print("Coluna criada: 'duration_min' a partir de 'duration_ms' (em minutos)")
else:
    print("Coluna 'duration_ms' não encontrada. Nenhuma conversão realizada.")

# ------------------------------------------------------------
# 3.4 Remoção de colunas irrelevantes
# ------------------------------------------------------------
print("\n[3.4] Remoção de colunas sem relevância analítica")
print("-" * 72)

cols_to_drop = [
    c for c in ["Unnamed: 0", "Unnamed: 0.1", "track_id"] if c in df_clean.columns
]

if cols_to_drop:
    df_clean.drop(columns=cols_to_drop, inplace=True)
    print(f"Colunas removidas: {cols_to_drop}")
else:
    print("Nenhuma coluna irrelevante encontrada para remoção.")

# ------------------------------------------------------------
# 3.5 Remoção por popularidade
# ------------------------------------------------------------
print("\n[3.5] Filtragem por popularidade")
print("-" * 72)

removidos_popularity = 0
if "popularity" in df_clean.columns:
    antes = len(df_clean)
    df_clean = df_clean[df_clean["popularity"] > 0].copy()
    removidos_popularity = antes - len(df_clean)
    print(f"Registros removidos com popularity = 0: {removidos_popularity}")
    print(f"Registros restantes: {len(df_clean)}")
else:
    print("Coluna 'popularity' não encontrada.")

# ------------------------------------------------------------
# 3.6 Validação de intervalos
# ------------------------------------------------------------
print("\n[3.6] Validação de intervalos conhecidos")
print("-" * 72)

print("\n[3.6] Validação de intervalos:")
validacoes = {
    "duration_ms": (
        (lambda df: df["duration_ms"] > 0)
        if "duration_ms" in df_clean.columns
        else None
    ),
    "popularity": (
        (lambda df: df["popularity"].between(0, 100))
        if "popularity" in df_clean.columns
        else None
    ),
    "danceability": (
        (lambda df: df["danceability"].between(0.0, 1.0))
        if "danceability" in df_clean.columns
        else None
    ),
    "energy": (
        (lambda df: df["energy"].between(0.0, 1.0))
        if "energy" in df_clean.columns
        else None
    ),
    "loudness": (
        (lambda df: df["loudness"].between(-60, 0))
        if "loudness" in df_clean.columns
        else None
    ),
    "speechiness": (
        (lambda df: df["speechiness"].between(0.0, 1.0))
        if "speechiness" in df_clean.columns
        else None
    ),
    "acousticness": (
        (lambda df: df["acousticness"].between(0.0, 1.0))
        if "acousticness" in df_clean.columns
        else None
    ),
    "instrumentalness": (
        (lambda df: df["instrumentalness"].between(0.0, 1.0))
        if "instrumentalness" in df_clean.columns
        else None
    ),
    "liveness": (
        (lambda df: df["liveness"].between(0.0, 1.0))
        if "liveness" in df_clean.columns
        else None
    ),
    "valence": (
        (lambda df: df["valence"].between(0.0, 1.0))
        if "valence" in df_clean.columns
        else None
    ),
    "tempo": (lambda df: df["tempo"] > 0) if "tempo" in df_clean.columns else None,
    "key": (lambda df: df["key"].between(0, 11)) if "key" in df_clean.columns else None,
    "mode": (
        (lambda df: df["mode"].isin([0, 1])) if "mode" in df_clean.columns else None
    ),
    "time_signature": (
        (lambda df: df["time_signature"].between(1, 7))
        if "time_signature" in df_clean.columns
        else None
    ),
}

mascara_valida = pd.Series(True, index=df_clean.index)
inconsistencias = []

for col, regra in validacoes.items():
    if regra is not None:
        validos = regra(df_clean)
        invalidos = (~validos).sum()
        if invalidos > 0:
            inconsistencias.append((col, invalidos))
        mascara_valida &= validos

if inconsistencias:
    print("Registros inválidos encontrados por coluna:")
    for col, qtd in inconsistencias:
        print(f" - {col:<18}: {qtd}")
else:
    print("Nenhuma inconsistência encontrada nos intervalos avaliados.")

antes = len(df_clean)
df_clean = df_clean[mascara_valida].copy()
removidos_invalidos = antes - len(df_clean)

print(f"Total de registros removidos por valores inválidos: {removidos_invalidos}")
print(f"Registros restantes: {len(df_clean)}")

# ------------------------------------------------------------
# 3.7 Tratamento de outliers via IQR
# ------------------------------------------------------------
print("\n[3.7] Tratamento de outliers por clipagem IQR")
print("-" * 72)

outlier_cols = [
    c
    for c in [
        "popularity",
        "danceability",
        "energy",
        "loudness",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "liveness",
        "valence",
        "tempo",
        "duration_min",
    ]
    if c in df_clean.columns
]

resumo_outliers = []

for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
    resumo_outliers.append((col, outliers, lower, upper))

print("Resumo da clipagem:")
for col, qtd, lower, upper in resumo_outliers:
    print(
        f" - {col:<18}: {qtd:>5} outliers ajustados | limites [{lower:.4f}, {upper:.4f}]"
    )

# ------------------------------------------------------------
# 3.8 Normalização Min-Max
# ------------------------------------------------------------
print("\n[3.8] Normalização Min-Max")
print("-" * 72)

features_to_normalize = [
    c
    for c in [
        "popularity",
        "danceability",
        "energy",
        "loudness",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "liveness",
        "valence",
        "tempo",
        "duration_min",
    ]
    if c in df_clean.columns
]

df_normalized = df_clean.copy()

if features_to_normalize:
    scaler = MinMaxScaler()
    df_normalized[features_to_normalize] = scaler.fit_transform(
        df_clean[features_to_normalize]
    )
    print("Colunas normalizadas:")
    for col in features_to_normalize:
        print(f" - {col}")
else:
    print("Nenhuma coluna elegível encontrada para normalização.")

# ------------------------------------------------------------
# Resumo final
# ------------------------------------------------------------
print("\n" + "═" * 72)
print("RESUMO DO PRÉ-PROCESSAMENTO")
print("═" * 72)

print(f"Registros iniciais                         : {total_registros_inicial}")
print(f"Duplicatas removidas                      : {removidos_duplicatas}")
print(f"Registros removidos por ausentes críticos : {removidos_ausentes}")
print(f"Registros removidos por popularity = 0    : {removidos_popularity}")
print(f"Registros removidos por valores inválidos : {removidos_invalidos}")
print(f"Registros finais (dados tratados)         : {len(df_clean)}")
print(f"Colunas finais (dados tratados)           : {df_clean.shape[1]}")
print(
    f"Dimensão final dos dados normalizados     : {df_normalized.shape[0]} x {df_normalized.shape[1]}"
)


════════════════════════════════════════════════════════════════════════
3. PRÉ-PROCESSAMENTO DOS DADOS
════════════════════════════════════════════════════════════════════════

Dimensão inicial do dataset: 114000 linhas x 22 colunas

[3.1] Remoção de duplicatas
------------------------------------------------------------------------
Critério utilizado: combinação de ['track_name', 'artists']
Registros removidos: 32656
Registros restantes: 81344

[3.2] Tratamento de valores ausentes
------------------------------------------------------------------------
Registros removidos por 3 ou mais valores ausentes: 1
Nenhuma imputação foi necessária após a remoção de registros críticos.
Valores ausentes restantes no dataset: 0

[3.3] Transformação de atributos
------------------------------------------------------------------------
Coluna criada: 'duration_min' a partir de 'duration_ms' (em minutos)

[3.4] Remoção de colunas sem relevância analítica
---------------------------------------------

## 4 - Análise Estatística Básica e Correlação

Nesta etapa, são calculadas medidas estatísticas descritivas das principais variáveis numéricas do dataset tratado, como média, mediana, desvio padrão, mínimo e máximo. Em seguida, é analisada a correlação entre os atributos para identificar relações lineares relevantes entre as características musicais.

Além da tabela estatística, também são gerados gráficos de distribuição, boxplots, matriz de correlação e visualizações complementares para apoiar a interpretação dos dados.

In [ ]:
# ============================================================
# 4. ANÁLISE ESTATÍSTICA BÁSICA E CORRELAÇÃO
# ============================================================

print("\n" + "═" * 72)
print("4. ESTATÍSTICAS DESCRITIVAS DOS DADOS TRATADOS")
print("═" * 72)

stats_cols = features_to_normalize
stats = df_clean[stats_cols].agg(["mean", "median", "std", "min", "max"])
stats.index = ["Média", "Mediana", "Desvio Padrão", "Mínimo", "Máximo"]

print(f"\nVariáveis analisadas: {len(stats_cols)}")
print("Colunas consideradas:")
for col in stats_cols:
    print(f" - {col}")

print("\n" + stats.round(4).T.to_string())


════════════════════════════════════════════════════════════════════════
4. ESTATÍSTICAS DESCRITIVAS DOS DADOS TRATADOS
════════════════════════════════════════════════════════════════════════

Variáveis analisadas: 11
Colunas consideradas:
 - popularity
 - danceability
 - energy
 - loudness
 - speechiness
 - acousticness
 - instrumentalness
 - liveness
 - valence
 - tempo
 - duration_min

                     Média   Mediana  Desvio Padrão   Mínimo    Máximo
popularity         37.1242   37.0000        17.6902   1.0000   90.5000
danceability        0.5624    0.5750         0.1745   0.0910    0.9850
energy              0.6407    0.6810         0.2545   0.0000    1.0000
loudness           -8.1733   -7.2360         4.1369 -18.1095   -0.0010
speechiness         0.0694    0.0493         0.0453   0.0221    0.1651
acousticness        0.3233    0.1850         0.3355   0.0000    0.9960
instrumentalness    0.0922    0.0001         0.1520   0.0000    0.3675
liveness            0.2039    0.1330  

### 4.1 - Gráficos

Os gráficos ajudam a visualizar a distribuição, dispersão e relação entre as variáveis do dataset. Observa-se que atributos como `danceability`, `energy` e `valence` se concentram em valores intermediários a altos, enquanto `speechiness` e `instrumentalness` aparecem mais concentradas em valores baixos.

A matriz de correlação e os gráficos de dispersão reforçam que algumas variáveis possuem associação entre si, como no caso de atributos ligados à intensidade sonora. Já os gráficos por gênero e por conteúdo explícito complementam a análise ao permitir comparar a popularidade média entre diferentes grupos.

In [ ]:
# ------------------------------------------------------------
# 4.1 Gráficos
# ------------------------------------------------------------
print("\n" + "-" * 72)
print("4.1 GERAÇÃO DOS GRÁFICOS")
print("-" * 72)

# Gráfico 2: Distribuição das variáveis numéricas
n_cols_plot = min(len(features_to_normalize), 12)
cols_plot = features_to_normalize[:n_cols_plot]
n_rows = (n_cols_plot + 2) // 3

fig, axes = plt.subplots(n_rows, 3, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(cols_plot):
    axes[i].hist(
        df_clean[col].dropna(),
        bins=30,
        color="steelblue",
        edgecolor="white",
        alpha=0.85,
    )
    axes[i].set_title(f"Distribuição: {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequência")
    mean_val = df_clean[col].mean()
    axes[i].axvline(
        mean_val,
        color="red",
        linestyle="--",
        linewidth=1.2,
        label=f"Média: {mean_val:.2f}",
    )
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    "Distribuição das Variáveis Numéricas", fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("resultados/02_distribuicoes.png", bbox_inches="tight")
print("Gráfico salvo em /resultados: 02_distribuicoes.png")

# Gráfico 3: Boxplots
fig, axes = plt.subplots(n_rows, 3, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(cols_plot):
    axes[i].boxplot(
        df_clean[col].dropna(),
        patch_artist=True,
        boxprops=dict(facecolor="lightblue", color="navy"),
        medianprops=dict(color="red", linewidth=2),
    )
    axes[i].set_title(f"Boxplot: {col}")
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots das Variáveis Numéricas", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("resultados/03_boxplots.png", bbox_inches="tight")
print("Gráfico salvo em /resultados: 03_boxplots.png")

# Gráfico 4: Matriz de correlação
corr_matrix = df_clean[stats_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    mask=mask,
    ax=ax,
    linewidths=0.5,
    annot_kws={"size": 8},
    vmin=-1,
    vmax=1,
)
ax.set_title(
    "Matriz de Correlação entre Variáveis Numéricas", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("resultados/04_correlacao.png", bbox_inches="tight")
print("Gráfico salvo em /resultados: 04_correlacao.png")

# Gráfico 5: Top correlações com 'popularity'
if "popularity" in stats_cols:
    corr_pop = corr_matrix["popularity"].drop("popularity").sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["salmon" if v < 0 else "steelblue" for v in corr_pop]
    corr_pop.plot(kind="barh", ax=ax, color=colors, edgecolor="black")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(
        "Correlação das Variáveis com Popularity", fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Coeficiente de Correlação")
    plt.tight_layout()
    plt.savefig("resultados/05_correlacao_popularity.png", bbox_inches="tight")
    print("Gráfico salvo em /resultados: 05_correlacao_popularity.png")

# Gráfico 6: Top 10 gêneros mais frequentes (se existir)
if "track_genre" in df_clean.columns:
    top_genres = df_clean["track_genre"].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(10, 5))
    top_genres.plot(kind="bar", ax=ax, color="mediumpurple", edgecolor="black")
    ax.set_title("Top 10 Gêneros Musicais", fontsize=12, fontweight="bold")
    ax.set_xlabel("Gênero")
    ax.set_ylabel("Quantidade de Faixas")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("resultados/06_top_generos.png", bbox_inches="tight")
    print("Gráfico salvo em /resultados: 06_top_generos.png")

# Gráfico 7: Scatter - Energy vs Popularity
if "energy" in df_clean.columns and "popularity" in df_clean.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sample = df_clean.sample(min(2000, len(df_clean)), random_state=42)
    ax.scatter(
        sample["energy"],
        sample["popularity"],
        alpha=0.3,
        color="teal",
        edgecolors="none",
        s=15,
    )
    ax.set_xlabel("Energy")
    ax.set_ylabel("Popularity")
    ax.set_title("Energy vs Popularity", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("resultados/07_energy_vs_popularity.png", bbox_inches="tight")
    print("Gráfico salvo em /resultados: 07_energy_vs_popularity.png")

# Gráfico 8: Danceability vs Valence
if "danceability" in df_clean.columns and "valence" in df_clean.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sample = df_clean.sample(min(2000, len(df_clean)), random_state=42)
    scatter = ax.scatter(
        sample["danceability"],
        sample["valence"],
        c=sample["popularity"] if "popularity" in sample.columns else "blue",
        cmap="viridis",
        alpha=0.4,
        s=15,
    )
    plt.colorbar(scatter, ax=ax, label="Popularity")
    ax.set_xlabel("Danceability")
    ax.set_ylabel("Valence")
    ax.set_title(
        "Danceability vs Valence (cor = Popularity)", fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("resultados/08_danceability_vs_valence.png", bbox_inches="tight")
    print("Gráfico salvo em /resultados: 08_danceability_vs_valence.png")

# Gráfico 9: Popularidade média por gênero
if "track_genre" in df_clean.columns and "popularity" in df_clean.columns:
    genre_popularity = (
        df_clean.groupby("track_genre")["popularity"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    genre_popularity.plot(kind="bar", ax=ax, color="cornflowerblue", edgecolor="black")
    ax.set_title(
        "Top 10 Gêneros por Popularidade Média", fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Gênero")
    ax.set_ylabel("Popularidade Média")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("resultados/09_popularidade_media_genero.png", bbox_inches="tight")
    print("Gráfico salvo em /resultados: 09_popularidade_media_genero.png")

# Gráfico 10: Popularidade média por explicit
if "explicit" in df_clean.columns and "popularity" in df_clean.columns:
    explicit_popularity = df_clean.groupby("explicit")["popularity"].mean()

    # Ajusta rótulos para leitura mais clara
    explicit_popularity.index = [
        "Não explícita" if v in [False, 0] else "Explícita"
        for v in explicit_popularity.index
    ]

    fig, ax = plt.subplots(figsize=(6, 4))
    explicit_popularity.plot(
        kind="bar", ax=ax, color=["mediumseagreen", "salmon"], edgecolor="black"
    )
    ax.set_title(
        "Popularidade Média por Conteúdo Explícito", fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Tipo de Faixa")
    ax.set_ylabel("Popularidade Média")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig("resultados/10_popularidade_media_explicit.png", bbox_inches="tight")
    print("Gráfico salvo em /resultados: 10_popularidade_media_explicit.png")

# ------------------------------------------------------------
# Resumo da correlação
# ------------------------------------------------------------
print("\n" + "-" * 72)
print("4.2 RESUMO DAS CORRELAÇÕES")
print("-" * 72)

corr_pairs = (
    corr_matrix.where(~np.eye(corr_matrix.shape[0], dtype=bool))
    .stack()
    .sort_values(key=np.abs, ascending=False)
)
corr_pairs = corr_pairs[~corr_pairs.index.duplicated()]

top_corr = corr_pairs.head(10)

print("Maiores correlações em valor absoluto:")
for (var1, var2), valor in top_corr.items():
    print(f" - {var1} x {var2}: {valor:.4f}")

if "popularity" in corr_matrix.columns:
    corr_pop_sorted = (
        corr_matrix["popularity"]
        .drop("popularity")
        .sort_values(key=np.abs, ascending=False)
    )
    print("\nCorrelações com popularity:")
    for var, valor in corr_pop_sorted.items():
        print(f" - {var:<18}: {valor:.4f}")


------------------------------------------------------------------------
4.1 GERAÇÃO DOS GRÁFICOS
------------------------------------------------------------------------
Gráfico salvo em /resultados: 02_distribuicoes.png
Gráfico salvo em /resultados: 03_boxplots.png
Gráfico salvo em /resultados: 04_correlacao.png
Gráfico salvo em /resultados: 05_correlacao_popularity.png
Gráfico salvo em /resultados: 06_top_generos.png
Gráfico salvo em /resultados: 07_energy_vs_popularity.png
Gráfico salvo em /resultados: 08_danceability_vs_valence.png
Gráfico salvo em /resultados: 09_popularidade_media_genero.png
Gráfico salvo em /resultados: 10_popularidade_media_explicit.png

------------------------------------------------------------------------
4.2 RESUMO DAS CORRELAÇÕES
------------------------------------------------------------------------
Maiores correlações em valor absoluto:
 - loudness x energy: 0.7824
 - energy x loudness: 0.7824
 - energy x acousticness: -0.7237
 - acousticness x energ

## 5 - Dataset Final

Nesta etapa, é apresentado o conjunto de dados resultante após o pré-processamento. São mostradas as dimensões finais da base tratada e da base normalizada, além de uma amostra dos registros obtidos e um resumo das transformações aplicadas ao longo do processo.

O objetivo é evidenciar como o dataset foi reorganizado para se tornar mais consistente, limpo e adequado para análise estatística.

In [ ]:
# ============================================================
# 5. DATASET FINAL
# ============================================================

print("\n" + "═" * 72)
print("5. DATASET FINAL")
print("═" * 72)

print("\n[5.1] Dimensões finais")
print("-" * 72)
print(f"Dados tratados     : {df_clean.shape[0]} linhas x {df_clean.shape[1]} colunas")
print(
    f"Dados normalizados : {df_normalized.shape[0]} linhas x {df_normalized.shape[1]} colunas"
)

total_removidos = (
    removidos_duplicatas
    + removidos_ausentes
    + removidos_popularity
    + removidos_invalidos
)

print("\n[5.2] Totalização de registros removidos")
print("-" * 72)
print(f"Registros iniciais                         : {total_registros_inicial}")
print(f"Duplicatas removidas                      : {removidos_duplicatas}")
print(f"Registros removidos por ausentes críticos : {removidos_ausentes}")
print(f"Registros removidos por popularity = 0    : {removidos_popularity}")
print(f"Registros removidos por valores inválidos : {removidos_invalidos}")
print(f"Total removido                            : {total_removidos}")
print(f"Registros finais                          : {len(df_clean)}")

print("\n[5.3] Amostra do dataset tratado")
print("-" * 72)
print(df_clean.head().to_string(index=False))

print("\n[5.4] Amostra das variáveis normalizadas")
print("-" * 72)
print(df_normalized[features_to_normalize].head().round(4).to_string(index=False))

# Salvar datasets
print("\n[5.5] Arquivos gerados")
print("-" * 72)

df_clean.to_csv("spotify_clean.csv", index=False)
df_normalized.to_csv("spotify_normalized.csv", index=False)

print("Dataset tratado salvo em: spotify_clean.csv")
print("Dataset normalizado salvo em: spotify_normalized.csv")

print("\n" + "═" * 72)
print("RESUMO DAS TRANSFORMAÇÕES REALIZADAS")
print("═" * 72)

transformacoes = [
    "1. Remoção de duplicatas com base em 'track_name' e 'artists'",
    "2. Tratamento de valores ausentes",
    "   - remoção de registros com 3 ou mais valores ausentes",
    "   - preenchimento de valores numéricos com mediana",
    "   - preenchimento de valores categóricos com moda",
    "3. Conversão de 'duration_ms' para 'duration_min'",
    "4. Remoção de colunas sem relevância analítica, como 'Unnamed: 0', 'Unnamed: 0.1' e 'track_id'",
    "5. Remoção de faixas com 'popularity = 0'",
    "6. Validação de intervalos conhecidos nas colunas numéricas",
    "7. Tratamento de outliers por clipagem com base no IQR",
    "8. Normalização Min-Max das principais variáveis numéricas",
]

for item in transformacoes:
    print(item)


════════════════════════════════════════════════════════════════════════
5. DATASET FINAL
════════════════════════════════════════════════════════════════════════

[5.1] Dimensões finais
------------------------------------------------------------------------
Dados tratados     : 75689 linhas x 20 colunas
Dados normalizados : 75689 linhas x 20 colunas

[5.2] Totalização de registros removidos
------------------------------------------------------------------------
Registros iniciais                         : 114000
Duplicatas removidas                      : 32656
Registros removidos por ausentes críticos : 1
Registros removidos por popularity = 0    : 5451
Registros removidos por valores inválidos : 203
Total removido                            : 38311
Registros finais                          : 75689

[5.3] Amostra do dataset tratado
------------------------------------------------------------------------
               artists                                             album_name 